# Day 16 Mini Capstone — Predict Customer Churn
## End-to-End Tabular Machine Learning

**Deliverable:** a notebook that runs top-to-bottom and ends with a business recommendation and reflection.

This is a standalone mini capstone. The dataset is generated in memory; no CSV download or internet connection is required.

### Business problem
Given a customer's account and plan features as of today, predict whether they will churn (`churn = 1`), so the retention team can target save-offers.

This is a binary-classification problem with an expected churn rate of approximately 27%. We will protect **recall on the churn class** because missed churners may be costly, and use **ROC-AUC** as the threshold-independent model-selection metric.

## Prerequisites — Generate the synthetic dataset

Run this cell once. It creates 4,000 telecom-style customer records and a deliberate post-outcome leakage feature, `exit_survey_sent`, which we will identify and remove in Step 2.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification

RNG = 42
rng = np.random.default_rng(RNG)

N = 4000

X_num, y = make_classification(
    n_samples=N,
    n_features=6,
    n_informative=5,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.76, 0.24],
    flip_y=0.13,
    class_sep=0.85,
    random_state=RNG
)

Xs = (X_num - X_num.mean(0)) / X_num.std(0)

df = pd.DataFrame()

# Numeric account features
df["tenure_months"] = np.clip(Xs[:, 4] * 12 + 34, 1, 72).round().astype(int)
df["monthly_charges"] = np.clip(Xs[:, 0] * 13 + 72, 18, 130).round(2)
df["support_tickets"] = np.clip(Xs[:, 2] * 1.7 + 3, 0, 25).round().astype(int)
df["avg_latency_ms"] = np.clip(Xs[:, 3] * 35 + 120, 20, 400).round().astype(int)
df["addons_count"] = np.clip(Xs[:, 1] * 1.3 + 4, 0, 8).round().astype(int)
df["days_since_login"] = np.clip(
    -Xs[:, 4] * 8 + 16 + rng.normal(0, 4, N), 0, 90
).round().astype(int)
df["total_charges"] = (df["monthly_charges"] * df["tenure_months"]).round(2)

# Categorical plan features
def by_class(p_churn, p_stay, choices):
    out = np.empty(N, dtype=object)
    m = (y == 1)
    out[m] = rng.choice(choices, m.sum(), p=p_churn)
    out[~m] = rng.choice(choices, (~m).sum(), p=p_stay)
    return out

df["contract_type"] = by_class(
    [0.75, 0.18, 0.07], [0.34, 0.35, 0.31],
    ["Month-to-month", "One year", "Two year"]
)
df["payment_method"] = by_class(
    [0.55, 0.20, 0.15, 0.10], [0.25, 0.30, 0.25, 0.20],
    ["Electronic check", "Mailed check", "Bank transfer", "Credit card"]
)
df["internet_service"] = by_class(
    [0.65, 0.28, 0.07], [0.40, 0.35, 0.25],
    ["Fiber optic", "DSL", "None"]
)
df["has_dependents"] = by_class(
    [0.82, 0.18], [0.60, 0.40], ["No", "Yes"]
)

# Deliberate leakage: survey is almost always sent after churn
df["exit_survey_sent"] = np.where(
    y == 1, rng.binomial(1, 0.97, N), rng.binomial(1, 0.02, N)
)

df["churn"] = y

print("dataset shape:", df.shape)
print("churn rate:", round(df["churn"].mean(), 3))
df.head()

## Step 1 — Frame the problem and explore (EDA)

We have a binary target: `0 = stay` and `1 = churn`. The class distribution is expected to be roughly 72.5% stay and 27.5% churn.

Recall answers: **Of all customers who actually churn, what fraction did the model identify?** We prioritise recall because missing a likely churner can be more expensive than contacting a customer who would have stayed. ROC-AUC will be used to compare models independently of a particular decision threshold.

In [ ]:
# Class balance
print(df["churn"].value_counts(normalize=True).round(3), "\\n")

# Numeric feature comparison
num_cols = [
    "tenure_months", "monthly_charges", "support_tickets",
    "avg_latency_ms", "addons_count", "days_since_login",
    "total_charges"
]
print(df.groupby("churn")[num_cols].mean().round(1).T, "\\n")

# Churn rate by contract type
print(
    df.groupby("contract_type")["churn"]
    .mean()
    .round(3)
    .sort_values(ascending=False)
)

# Visualise tenure vs churn and churn rate by contract
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for k, lbl in [(0, "stay"), (1, "churn")]:
    ax[0].hist(
        df.loc[df.churn == k, "tenure_months"],
        bins=25, alpha=0.6, label=lbl
    )
ax[0].set_title("Tenure by outcome")
ax[0].set_xlabel("tenure_months")
ax[0].legend()

df.groupby("contract_type")["churn"].mean().sort_values().plot.barh(ax=ax[1])
ax[1].set_title("Churn rate by contract")
ax[1].set_xlabel("churn rate")

plt.tight_layout()
plt.show()

### Step 1 checkpoint

Expected broad patterns:
- Approximately 72.5% stayers and 27.5% churners.
- Churners tend to have lower tenure, more support tickets, and more days since their last login.
- Month-to-month contracts have the highest churn rate; two-year contracts have the lowest.

These are associations in this synthetic dataset, not proof that a feature causes churn.

## Step 2 — Split train / validation / test, and hunt for leakage

We create three stratified partitions:
- **Training (70%)**: model fitting and cross-validation.
- **Validation (15%)**: a held-out check during development and feature interpretation.
- **Test (15%)**: final evaluation, held back until Step 6.

A feature leaks information when it contains information that would not be available at the time a prediction is made. `exit_survey_sent` is a post-outcome flag, so it must be removed. `total_charges` stays because it is calculated from account information available today.

In [ ]:
from sklearn.model_selection import train_test_split

# Leakage check #1: inspect unusually high numeric correlations
leak_scores = (
    df.corr(numeric_only=True)["churn"]
    .drop("churn")
    .abs()
    .sort_values(ascending=False)
)
print("abs. correlation with churn:\\n", leak_scores.round(3), "\\n")

suspects = leak_scores[leak_scores > 0.9].index.tolist()
print("LEAK suspects (corr > 0.9):", suspects)

# Drop the post-outcome leakage feature(s)
df_clean = df.drop(columns=suspects)

X = df_clean.drop(columns=["churn"])
y = df_clean["churn"]

# 70% train / 15% validation / 15% test
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RNG
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RNG
)

print("\\nshapes:", X_train.shape, X_val.shape, X_test.shape)
print(
    "churn rate train/val/test:",
    round(y_train.mean(), 3),
    round(y_val.mean(), 3),
    round(y_test.mean(), 3)
)

### Step 2 checkpoint

Expected shapes:
- `X_train`: `(2800, 11)`
- `X_val`: `(600, 11)`
- `X_test`: `(600, 11)`

The churn rates should remain close across the three splits because `stratify` preserves the class proportions. Do not use `X_test` or `y_test` to choose models, hyperparameters, or a threshold.

## Step 3 — Feature engineering + scaling inside a Pipeline

Numeric and categorical features need different preprocessing:
- Numeric columns: `StandardScaler` standardises their scale.
- Categorical columns: `OneHotEncoder` converts categories into indicator columns.
- `ColumnTransformer` applies the appropriate transformation to each group.
- `Pipeline` keeps preprocessing and model fitting together, so cross-validation fits preprocessing only on each training fold.

We fit the standalone transformer on `X_train` only for a shape sanity-check. The model pipelines below will independently fit their preprocessing within each fold.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_features = X_train.select_dtypes("number").columns.tolist()
cat_features = X_train.select_dtypes("object").columns.tolist()

print("numeric:", num_features)
print("categorical:", cat_features)

pre = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

# Sanity-check transform width (fit on TRAIN ONLY)
Z = pre.fit_transform(X_train)
print("\\ntransformed matrix:", Z.shape)
print("feature count after one-hot:", len(pre.get_feature_names_out()))

### Step 3 checkpoint

Expected transformed shape: `(2800, 19)` — 7 numeric columns plus 12 one-hot columns. `handle_unknown="ignore"` allows the pipeline to handle categories not seen during fitting.

## Step 4 — Train and compare four model families

We compare Logistic Regression, Random Forest, Gradient Boosting, and an RBF Support Vector Machine using the same preprocessing and stratified 5-fold cross-validation. ROC-AUC is the comparison metric. `class_weight="balanced"` is used for the models that support it to account for the minority churn class.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

models = {
    "LogReg": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, random_state=RNG, class_weight="balanced"
    ),
    "GradBoost": GradientBoostingClassifier(random_state=RNG),
    "SVM_RBF": SVC(
        kernel="rbf", probability=True,
        class_weight="balanced", random_state=RNG
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
results = {}

for name, clf in models.items():
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    scores = cross_val_score(
        pipe, X_train, y_train, cv=cv, scoring="roc_auc"
    )
    results[name] = scores
    print(
        f"{name:13s} ROC-AUC = "
        f"{scores.mean():.3f} +/- {scores.std():.3f}"
    )

### Step 4 checkpoint

The assignment expects all four models to be close, around ROC-AUC 0.94. The suggested model to carry forward is Gradient Boosting; Logistic Regression remains an interpretable baseline. Small differences can occur across library versions.

## Step 5 — Tune the winner with GridSearchCV

We search a small grid for Gradient Boosting using only the training partition and the same cross-validation strategy. The validation partition is used for a confirmatory ROC-AUC check. The test partition remains untouched.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

gb_pipe = Pipeline([
    ("pre", pre),
    ("clf", GradientBoostingClassifier(random_state=RNG))
])

param_grid = {
    "clf__n_estimators": [150, 300],
    "clf__learning_rate": [0.05, 0.1],
    "clf__max_depth": [2, 3],
}

grid = GridSearchCV(
    gb_pipe, param_grid, cv=cv,
    scoring="roc_auc", n_jobs=-1
)
grid.fit(X_train, y_train)

print("best params:", grid.best_params_)
print("best CV ROC-AUC:", round(grid.best_score_, 3))

best_model = grid.best_estimator_

val_auc = roc_auc_score(
    y_val, best_model.predict_proba(X_val)[:, 1]
)
print("validation ROC-AUC:", round(val_auc, 3))

### Step 5 checkpoint

The assignment's approximate checkpoint is a shallow Gradient Boosting model (`max_depth=2`), with best CV and validation ROC-AUC around 0.94. The exact best parameters can vary slightly by library version and generated data details.

## Step 6 — Evaluate with the right metrics + choose a threshold

Now we evaluate the frozen model on the test set. This is the first and only use of the test set for final performance evaluation.

At threshold 0.50, probabilities at or above 0.50 are classified as churn. Lowering the threshold generally catches more churners (higher recall), but can also increase false alarms (lower precision).

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, precision_score, recall_score, f1_score
)

proba = best_model.predict_proba(X_test)[:, 1]
pred_050 = (proba >= 0.50).astype(int)

print("TEST ROC-AUC:", round(roc_auc_score(y_test, proba), 3), "\\n")
print(classification_report(
    y_test, pred_050, target_names=["stay", "churn"], digits=3
))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred_050),
    display_labels=["stay", "churn"]
).plot(values_format="d")
plt.title("Confusion matrix @ threshold 0.50")
plt.show()

# Business trade-off: sweep decision thresholds
print(f"{'thr':>5} {'precision':>10} {'recall':>8} {'f1':>6}")
for t in [0.50, 0.40, 0.35, 0.30, 0.25]:
    p = (proba >= t).astype(int)
    print(
        f"{t:5.2f} {precision_score(y_test, p):10.3f} "
        f"{recall_score(y_test, p):8.3f} {f1_score(y_test, p):6.3f}"
    )

### Step 6 interpretation

- **True positive (TP):** a customer who churns and is correctly flagged.
- **False positive (FP):** a customer who stays but is flagged for a save-offer.
- **False negative (FN):** a customer who churns but is missed.
- **True negative (TN):** a customer who stays and is correctly left unflagged.

A lower threshold may be appropriate when the cost of losing a customer is much greater than the cost of contacting a customer who would have stayed. The assignment suggests considering a threshold around 0.30, but the threshold should ultimately be justified using actual business costs. The test-set threshold sweep here is descriptive; for a rigorous deployment decision, choose the operating threshold on validation data and reserve the test set for one final evaluation.

## Step 7 — Interpret the model

Permutation importance measures how much validation ROC-AUC falls when one feature is shuffled. A larger drop means the model relied more on that feature for its predictions.

We also fit a Logistic Regression baseline and inspect its signed coefficients. Positive coefficients push the model's churn prediction upward; negative coefficients push it downward. Coefficient magnitude depends on the transformed feature scale and should not be interpreted as causal impact.

In [ ]:
from sklearn.inspection import permutation_importance

# 1) Permutation importance of tuned Gradient Boosting on validation data
perm = permutation_importance(
    best_model, X_val, y_val,
    n_repeats=8, random_state=RNG, scoring="roc_auc"
)

imp = pd.Series(
    perm.importances_mean, index=X.columns
).sort_values(ascending=False)

print("Permutation importance (ROC-AUC drop when shuffled):")
print(imp.round(4), "\\n")

# 2) Signed drivers from an interpretable Logistic Regression baseline
logreg = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
]).fit(X_train, y_train)

coef = pd.Series(
    logreg.named_steps["clf"].coef_[0],
    index=logreg.named_steps["pre"].get_feature_names_out()
)

print("Most churn-DRIVING (positive coef):\\n")
print(coef.sort_values().tail(4).round(2), "\\n")

print("Most PROTECTIVE (negative coef):\\n")
print(coef.sort_values().head(4).round(2))

imp.sort_values().plot.barh(
    figsize=(7, 4), title="Permutation importance"
)
plt.tight_layout()
plt.show()

### Step 7 interpretation

The assignment expects important signals to include tenure / total charges, contract type, and support-ticket count. The Logistic Regression coefficients should help describe directions, such as month-to-month contracts and support-ticket counts pushing churn risk upward, and longer tenure / two-year contracts pushing it downward.

Use your actual printed results when writing the final recommendation. Importance and coefficients describe model associations, not proof of causation.

## Step 8 — Write the recommendation

The following is a suggested 3–5 sentence recommendation based on the assignment's expected checkpoint. Before submission, check that the figures and top drivers match your actual scorecard output.

**Recommendation (edit to match your actual results):**

We can predict churn with a test ROC-AUC of approximately 0.95 using account and plan information available today. At an operating threshold around 0.30, the model is expected to identify approximately 84% of customers who will leave, although some customers who would have stayed will also receive a retention contact. The strongest actionable signals include shorter tenure, month-to-month contracts, and higher support-ticket counts. We recommend piloting proactive support and an annual-contract incentive for at-risk month-to-month customers, while monitoring retention lift and the cost of unnecessary offers.

In [ ]:
summary = {
    "primary_metric": "recall on churn (ROC-AUC for selection)",
    "chosen_model": "GradientBoosting (tuned)",
    "test_roc_auc": round(roc_auc_score(y_test, proba), 3),
    "operating_threshold": 0.30,
}

p30 = (proba >= 0.30).astype(int)
summary["recall_at_0.30"] = round(recall_score(y_test, p30), 3)
summary["precision_at_0.30"] = round(precision_score(y_test, p30), 3)
summary["top_drivers"] = imp.head(3).index.tolist()

for k, v in summary.items():
    print(f"{k:22s}: {v}")

## Reflection — Answer each in 2–3 sentences

### 1. Leakage
Besides `exit_survey_sent`, name one other feature in a real churn table that could leak the outcome, and explain how you would detect it before trusting a suspiciously high score.

**Sample answer:** A cancellation-confirmation timestamp or a closed-account status could leak churn because it is only recorded after the customer has decided to leave. I would check when the field becomes available, inspect its relationship with the target, and verify that it would exist at the exact prediction time; I would remove it if it is post-outcome information.

### 2. Metric choice
You optimised ROC-AUC for selection but used a recall-oriented threshold. When would precision matter more than recall for a churn model? Give a concrete business scenario.

**Sample answer:** Precision matters more when retention outreach is expensive or disruptive, such as when each flagged customer receives a costly personal call or a large discount. In that situation, the company may prefer fewer but more reliable alerts, even if it misses some customers who eventually churn.

### 3. Trust
The four models scored within approximately 0.005 ROC-AUC of each other. Which would you deploy and why, and what does that tell you about where the next improvement should come from?

**Sample answer:** I would consider deploying Gradient Boosting because it performed competitively and provides feature-importance tools, while also comparing its operational cost with the simpler Logistic Regression baseline. Since the four models are so close, better customer-behaviour features, cleaner labels, and more representative data may improve performance more than switching algorithms.

### 4. Honesty
You touched the test set only in Step 6. Explain what would have gone wrong if you had used the test set to pick the threshold in Step 6 and reported that same number as your final result.

**Sample answer:** Choosing a threshold based on test-set performance would make the test set part of model development rather than an independent evaluation. The reported result would be optimistically biased because the threshold was selected to perform well on those particular test examples, so it would not provide an honest estimate of performance on new customers.

## Final submission checklist

- [ ] Save the notebook as `day16_capstone.ipynb`.
- [ ] Run **Kernel → Restart Kernel and Run All** (or the equivalent in your JupyterLab version).
- [ ] Confirm every cell runs top-to-bottom without errors.
- [ ] Check that Step 8's recommendation matches your actual scorecard.
- [ ] Review and personalise all four reflection answers.
- [ ] Submit the completed notebook.

**Important:** The assignment's threshold sweep is shown on the test set because that is how its supplied instructions are written. In a stricter production workflow, select the threshold using validation data, then evaluate that fixed threshold once on the test set.